# T05 — Prompts로 도메인 전문성 외부화

**학습 목표**
- `@mcp.prompt()` 데코레이터로 재사용 가능한 프롬프트 템플릿 정의
- `base.UserMessage`, `base.AssistantMessage`로 few-shot 예제 구성
- KDS 41 17 00 기준 구조 검토를 LLM-호환 프롬프트로 캡슐화
- Claude API와 통합해 실제 검토 응답 받기

**Prerequisites**
- T01-T04 완료 (Tools + Resources 등록 경험)
- `.env`에 `ANTHROPIC_API_KEY` 설정

> [!ref] 강의 노트: `Week_07.md` §2.4-§2.5 (Prompts)

## §0. 강의노트 매핑

> 📖 **Week_07.md §2.4-§2.5** (Prompts)

| 측면 | Skilljar 원본 (`skilljar/S6_05`) | 본 튜토리얼 (T05) |
|---|---|---|
| 도메인 | 일반 (`format` 프롬프트) | 한국 건축구조 (KDS 검토) |
| 프롬프트 수 | 1-3개 | **5개** (확장형) |
| 메시지 타입 | `UserMessage` | `UserMessage` + `AssistantMessage` (few-shot) |
| Claude API 통합 | 선택 | 필수 (실제 응답 검증) |

## §1. Setup

In [ ]:
# ── Setup ──────────────────────────────────────────────
import json
import asyncio
import anthropic
from dotenv import load_dotenv
from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

load_dotenv()

mcp = FastMCP("TutorialMCP", log_level="ERROR")
MODEL = "claude-haiku-4-5"

print("Setup 완료 — MCP + Anthropic SDK")

## §2. Prompts란?

MCP의 3대 기능 중 **사용자가 직접 선택**하는 기능. 코드 리뷰, 구조 검토, 비용 산정 등 도메인 전문 워크플로를 캡슐화.

| 기능 | 제어 주체 | 비유 |
|------|----------|------|
| Tools | 모델 (LLM) | AI가 알아서 도구 선택 |
| Resources | 애플리케이션 | 개발자가 데이터 제공 |
| **Prompts** | **사용자** | **사용자가 템플릿 선택** |

> [!finding] 왜 외부화하는가?
> KDS 조문 기반 검토 워크플로를 매번 직접 입력하지 않고, 한 번 정의해 재사용. 도메인 전문가가 프롬프트만 다듬으면 모든 사용자가 혜택.

## §3. 프롬프트 1 — 일반 코드 리뷰

도메인에 무관한 범용 프롬프트. 학습 시작점.

In [ ]:
@mcp.prompt()
def review_code(code: str, language: str = "python") -> str:
    """코드 리뷰를 수행하는 프롬프트 템플릿.

    Args:
        code: 리뷰할 코드
        language: 프로그래밍 언어
    """
    return f"""당신은 시니어 {language} 개발자입니다.
다음 코드를 리뷰하고 개선점을 제시해주세요:

```{language}
{code}
```

다음 관점에서 검토해주세요:
1. 코드 품질과 가독성
2. 잠재적 버그
3. 성능 최적화
4. Best practices 준수 여부"""

print("review_code 등록 완료")

## §4. 프롬프트 2 — KDS 구조 검토 (도메인 특화)

RC 부재의 KDS 41 17 00 기준 검토 워크플로 캡슐화.

In [ ]:
@mcp.prompt()
def structural_review(
    member_type: str,
    width: float,
    depth: float,
    fck: float,
    fy: float
) -> str:
    """KDS 41 17 00 기준 RC 부재 설계 검토 프롬프트.

    Args:
        member_type: 부재 종류 (beam, column, slab)
        width: 부재 폭 b (mm)
        depth: 부재 유효깊이 d (mm)
        fck: 콘크리트 설계기준 압축강도 (MPa)
        fy: 철근 항복강도 (MPa)
    """
    return f"""당신은 건축구조 전문가입니다. KDS 41 17 00 기준에 따라 다음 RC {member_type}의 설계를 검토해주세요.

## 부재 제원
- 종류: {member_type}
- 폭 (b): {width} mm
- 유효깊이 (d): {depth} mm
- fck: {fck} MPa
- fy: {fy} MPa

## 검토 항목
1. 최소/최대 철근비 확인 (KDS 41 17 00 §4.3)
2. 휨 강도 검토
3. 전단 강도 검토 (해당 시)
4. 처짐 검토 (해당 시)
5. 구조 상세 적정성"""

print("structural_review 등록 완료")

## §5. 프롬프트 3 — 설계 기준 체크리스트

In [ ]:
@mcp.prompt()
def design_check(standard: str = "KDS 41 17 00") -> str:
    """설계 기준 체크리스트 생성 프롬프트."""
    return f"""당신은 구조설계 검토자입니다. {standard} 기준에 따른 설계 체크리스트를 작성해주세요.

체크리스트에 포함할 항목:
1. 재료 기준 확인
2. 하중 조합 검토
3. 단면 설계 적정성
4. 상세 설계 확인
5. 시공성 검토"""

print("design_check 등록 완료")

## §6. 프롬프트 4 — 비용 추정 (신규 추가)

한국 건설시장 단가 기반 RC 부재 개략 비용 산정.

In [ ]:
@mcp.prompt(name="cost_estimate", description="RC 부재 개략 비용 추정")
def cost_estimate(
    member_type: str = Field(description="부재 종류"),
    volume: float = Field(description="콘크리트 체적 (m^3)"),
    rebar_ton: float = Field(description="철근량 (ton)")
) -> list[base.Message]:
    """한국 시장 단가 기반 비용 추정 프롬프트."""
    prompt = f"""한국 건설시장 단가 기준으로 다음 RC {member_type}의 개략 비용을 추정하세요.

## 입력 정보
- 부재 종류: {member_type}
- 콘크리트 체적: {volume} m^3 (C27 기준 단가 약 130,000원/m^3)
- 철근량: {rebar_ton} ton (SD400 기준 약 1,200,000원/ton)
- 거푸집·인건비 포함

## 출력 양식
1. 항목별 비용 (콘크리트 / 철근 / 거푸집 / 인건비)
2. 시공·관리비 5% 별도 가산
3. 총 비용 (원 단위)"""
    return [base.UserMessage(prompt)]

print("cost_estimate 등록 완료")

## §7. 프롬프트 5 — Few-shot 예제 (`summarize_review`)

`base.AssistantMessage`로 모범 답변 1개를 보여주면 LLM이 형식을 학습. 임원 보고용 3줄 요약.

In [ ]:
@mcp.prompt(name="summarize_review", description="구조 검토 결과를 임원 보고용 3줄로 요약")
def summarize_review(
    review_text: str = Field(description="검토 본문")
) -> list[base.Message]:
    """Few-shot 학습 기반 요약 프롬프트."""
    return [
        base.UserMessage(
            "다음 검토를 3줄 임원 보고로 요약해주세요:\n"
            "'300x600 RC 보에 대한 휨 검토 결과 KDS 41 17 00 §4.3 기준 만족. "
            "DCR=0.85로 양호하나 전단철근 간격이 150mm로 다소 큼. "
            "100mm 간격 보강 권장.'"
        ),
        base.AssistantMessage(
            "- 부재: B1 보 / 단면: 300x600\n"
            "- KDS 41 17 00 §4.3 휨 OK (DCR 0.85)\n"
            "- 권고: 전단철근 간격 100mm로 보강 검토"
        ),
        base.UserMessage(f"이번엔 다음을 같은 형식으로 요약해주세요:\n{review_text}"),
    ]

print("summarize_review 등록 완료 (few-shot)")

> ☑ **체크포인트 1**: 5개 프롬프트 등록
>
> `review_code`, `structural_review`, `design_check`, `cost_estimate`, `summarize_review`

## §8. 프롬프트 검증 — list / get

In [ ]:
async def demo_prompts():
    # 1. 프롬프트 목록
    prompts = await mcp.list_prompts()
    print(f"=== 등록된 프롬프트 ({len(prompts)}개) ===")
    for p in prompts:
        print(f"  - {p.name}: {p.description}")
        if p.arguments:
            for arg in p.arguments:
                req = "(필수)" if arg.required else "(선택)"
                print(f"      {arg.name} {req}")
        print()

    # 2. structural_review 호출
    print("=== structural_review 메시지 출력 ===")
    result = await mcp.get_prompt(
        "structural_review",
        arguments={
            "member_type": "beam",
            "width": "300",
            "depth": "540",
            "fck": "27",
            "fy": "400"
        }
    )
    for msg in result.messages:
        print(f"[{msg.role}]")
        text = msg.content.text if hasattr(msg.content, 'text') else str(msg.content)
        print(text[:400] + "..." if len(text) > 400 else text)
        print()

await demo_prompts()

> ☑ **체크포인트 2**: `list_prompts()`로 5개 모두 표시 확인
>
> `get_prompt()`로 메시지 객체가 반환되어야 함.

## §9. Claude API 통합 — 프롬프트로 실제 검토 받기

MCP 프롬프트의 메시지를 Claude API에 그대로 전달.

In [ ]:
api = anthropic.Anthropic()

async def run_prompt(prompt_name: str, args: dict) -> str:
    """MCP 프롬프트 → Claude API 실행 헬퍼."""
    result = await mcp.get_prompt(prompt_name, arguments=args)
    messages = [
        {
            "role": m.role,
            "content": m.content.text if hasattr(m.content, 'text') else str(m.content)
        }
        for m in result.messages
    ]
    resp = api.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=messages
    )
    return resp.content[0].text

# 실행: 300x540 RC 보 검토
answer = await run_prompt("structural_review", {
    "member_type": "beam",
    "width": "300",
    "depth": "540",
    "fck": "27",
    "fy": "400"
})
print("=== Claude 검토 결과 ===")
print(answer[:1500])

> ☑ **체크포인트 3**: Claude가 KDS 기준 휨/전단 검토 응답 생성
>
> `--persona-architect`나 도메인 전문가 검토 후 fine-tuning 가능.

## §10. Skilljar `format` 프롬프트와의 비교

| 측면 | Skilljar (`skilljar/S6_05`) | Tutorial (T05) |
|---|---|---|
| 프롬프트 개수 | 1-3개 (`format` 등 일반) | 5개 (`review_code` + KDS 4종) |
| 메시지 타입 | `UserMessage` 단일 | `UserMessage` + `AssistantMessage` (few-shot) |
| Field 메타데이터 | str return | `pydantic.Field` description |
| 도메인 깊이 | 일반 텍스트 변환 | KDS 41 17 00 조문 인용 |
| Claude 통합 | 선택 | 실행 가능 (API 호출) |

> [!tip] **언제 few-shot을 쓰는가?**
> 출력 형식이 중요한 경우 (요약 형식, JSON 키 순서 등). 단순 검토는 zero-shot으로 충분.

## §11. `tutorial_server.py` 갱신 저장

T01 도구 + T04 리소스 + T05 프롬프트 모두 통합.

In [ ]:
server_code = '''# tutorial_server.py — T01 도구 + T04 리소스 + T05 프롬프트 통합
import json
from datetime import datetime
from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

mcp = FastMCP("TutorialMCP")

# ── Tools ──────────────────────────────────────────
@mcp.tool()
def get_current_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """현재 시간 반환."""
    return datetime.now().strftime(format)

@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    """두 숫자 합."""
    return a + b

# ── Resources ──────────────────────────────────────
@mcp.resource("data://materials/concrete")
def get_concrete() -> str:
    return json.dumps({"C24": {"fck": 24}, "C27": {"fck": 27}}, indent=2)

# ── Prompts ────────────────────────────────────────
@mcp.prompt()
def structural_review(member_type: str, width: float, depth: float, fck: float, fy: float) -> str:
    return f"KDS 41 17 00 기준 {member_type} 검토: b={width}, d={depth}, fck={fck}, fy={fy}"

@mcp.prompt(name="cost_estimate")
def cost_estimate(member_type: str, volume: float, rebar_ton: float) -> list[base.Message]:
    return [base.UserMessage(f"한국 단가 기준 {member_type} 비용 추정: 콘크리트 {volume}m^3, 철근 {rebar_ton}ton")]

if __name__ == "__main__":
    mcp.run()
'''

with open("tutorial_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("tutorial_server.py 저장 완료 (도구 2 + 리소스 1 + 프롬프트 2)")

## §12. 다음 단계

**T06 Capstone**: 학생이 자율 도메인을 선택해 도구 3 + 리소스 2 + 프롬프트 1로 자신만의 MCP 서버 구축.

> [!ref] **현재 진행 상황**
> - ✅ T01: Tools 정의
> - ✅ T02: Inspector 검증
> - ✅ T03: SimpleMCPClient
> - ✅ T04: Resources
> - ✅ T05: Prompts (현재 노트북)
> - ⏳ T06: Capstone